In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import xy
from scipy.interpolate import RegularGridInterpolator
import matplotlib.pyplot as plt


In [ ]:
# ---------- utilidades ----------
def ensure_lat_ascending(lat_vec, grid2d):
    """Garante lat crescente e alinha a matriz ao vetor."""
    if lat_vec[0] > lat_vec[-1]:
        return lat_vec[::-1], grid2d[::-1, :]
    return lat_vec, grid2d

def read_raster_latlon(path):
    """Lê GeoTIFF (banda 1) -> (arr, lat_vec, lon_vec, crs), com NoData -> NaN e lat crescente."""
    with rasterio.open(path) as ds:
        arr = ds.read(1, masked=True).filled(np.nan).astype(float)
        transform = ds.transform
        crs = ds.crs
        nrows, ncols = ds.height, ds.width

    # coords nos centros de pixel
    lon_vec = np.array(xy(transform, [0]*ncols, list(range(ncols)), offset='center')[0])
    lat_vec = np.array(xy(transform, list(range(nrows)), [0]*nrows, offset='center')[1])

    # lat crescente para a interpoladora
    lat_vec, arr = ensure_lat_ascending(lat_vec, arr)
    return arr, lat_vec, lon_vec, crs

def load_rn_latlon(path):
    """Lê lat/lon (duas primeiras colunas). Ignora %, aceita ; , ou espaços, vírgula decimal."""
    df = pd.read_csv(
        path, engine="python", comment="%", sep=r"[;\s,]+",
        header=None, usecols=[0,1], names=["lat","lon"], skip_blank_lines=True
    )
    df["lat"] = pd.to_numeric(df["lat"].astype(str).str.replace(",", "."), errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"].astype(str).str.replace(",", "."), errors="coerce")
    df = df.dropna()
    if df.empty:
        raise ValueError("RNs: não encontrei lat/lon válidos.")
    return df

def load_rn_h_hn(path):
    """Lê lat, lon, h, HN e cria zeta_RN = h - HN."""
    df = pd.read_csv(
        path, engine="python", comment="%", sep=r"[;\s,]+",
        header=None, usecols=[0,1,2,3], names=["lat","lon","h","HN"], skip_blank_lines=True
    )
    for c in ["lat","lon","h","HN"]:
        df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", "."), errors="coerce")
    df = df.dropna()
    if df.empty:
        raise ValueError("RNs: não encontrei lat/lon/h/HN válidos.")
    df["zeta_RN"] = df["h"] - df["HN"]
    return df

In [ ]:
# ---------- caminhos (a partir de code/) ----------
p_geoid  = Path("../data/raw/Modelo_geoidal.tif")
p_quasi  = Path("../data/raw/Modelo_quase_geoide.tif")
p_rn     = Path("../data/raw/Pontos_RN_GPS.txt")
p_outN   = Path("../data/processed/N_model_RN.csv")
p_outz   = Path("../data/processed/validacao_quase_geoide.csv")

In [ ]:
# ---------- A) Extração N_model em RNs (geoide) ----------
arrN, latN_vec, lonN_vec, crsN = read_raster_latlon(p_geoid)
rn_xy = load_rn_latlon(p_rn)
print(f"Geoide CRS: {crsN}")

interpN = RegularGridInterpolator(
    (latN_vec, lonN_vec), arrN, method="linear",
    bounds_error=False, fill_value=np.nan
)
N_model = interpN(rn_xy[["lat","lon"]].to_numpy())

outN = rn_xy.copy()
outN["N_model"] = N_model
outN.to_csv(p_outN, index=False)
print(f"[OK] salvo: {p_outN.resolve()}  | pontos: {len(outN)}")

# ---------- B) Validação no quase-geoide (zeta = h - HN) ----------
arrZ, latZ_vec, lonZ_vec, crsZ = read_raster_latlon(p_quasi)
rn_zet = load_rn_h_hn(p_rn)
print(f"Quase-geoide CRS: {crsZ}")

interpZ = RegularGridInterpolator(
    (latZ_vec, lonZ_vec), arrZ, method="linear",
    bounds_error=False, fill_value=np.nan
)
rn_zet["zeta_model"] = interpZ(rn_zet[["lat","lon"]].to_numpy())

rn_zet["resid"] = rn_zet["zeta_model"] - rn_zet["zeta_RN"]
viés  = rn_zet["resid"].mean()
sigma = rn_zet["resid"].std(ddof=1)
rmse  = np.sqrt((rn_zet["resid"]**2).mean())
n     = rn_zet["resid"].notna().sum()

print(f"Validação (ζ): n={n} | viés={viés:.3f} m | σ={sigma:.3f} m | RMSE={rmse:.3f} m")
rn_zet.to_csv(p_outz, index=False)
print(f"[OK] salvo: {p_outz.resolve()}")


In [ ]:


res = rn_zet["resid"].dropna().to_numpy()
bias  = res.mean()
sigma = res.std(ddof=1)
rmse  = np.sqrt((res**2).mean())

plt.figure(figsize=(7,4))
plt.hist(res, bins=20, edgecolor="k")
plt.axvline(bias, linestyle="--", linewidth=1)
plt.xlabel("ζ_model − ζ_RN (m)")
plt.ylabel("Contagem")
plt.title(f"Resíduos em RNs — viés={bias:.3f} m, σ={sigma:.3f} m, RMSE={rmse:.3f} m")
plt.tight_layout()
plt.savefig("../data/processed/validacao_zeta_hist.png", dpi=300)
plt.show()

lon = rn_zet["lon"].to_numpy()
lat = rn_zet["lat"].to_numpy()
res = rn_zet["resid"].to_numpy()

# escala simétrica em torno de zero
v = np.nanmax(np.abs(res))
plt.figure(figsize=(7.5,6))
sc = plt.scatter(lon, lat, c=res, vmin=-v, vmax=+v, s=25)
plt.colorbar(sc, label="ζ_model − ζ_RN (m)")
plt.xlabel("Longitude (°)"); plt.ylabel("Latitude (°)")
plt.title("Resíduos do quase-geoide nos RNs")
plt.tight_layout()
plt.savefig("../data/processed/validacao_zeta_mapa.png", dpi=300)
plt.show()

z_obs = rn_zet["zeta_RN"].to_numpy()
z_mod = rn_zet["zeta_model"].to_numpy()

# R²
ss_res = np.nansum((z_obs - z_mod)**2)
ss_tot = np.nansum((z_obs - np.nanmean(z_obs))**2)
R2 = 1 - ss_res/ss_tot if ss_tot > 0 else np.nan

lims = [np.nanmin([z_obs,z_mod]), np.nanmax([z_obs,z_mod])]

plt.figure(figsize=(6,6))
plt.scatter(z_obs, z_mod, s=28, alpha=0.8)
plt.plot(lims, lims, 'k--', lw=1)  # 1:1
plt.xlabel("ζ_RN (m)")
plt.ylabel("ζ_model (m)")
plt.title(f"ζ_model vs ζ_RN — R²={R2:.3f} | viés={np.nanmean(z_mod - z_obs):.3f} m")
plt.tight_layout()
plt.savefig("../data/processed/validacao_zeta_scatter.png", dpi=300)
plt.show()

fig, ax = plt.subplots(1,2, figsize=(11,4), sharey=True)
ax[0].scatter(rn_zet["lat"], rn_zet["resid"], s=20)
ax[0].set_xlabel("Latitude (°)"); ax[0].set_ylabel("ζ_model − ζ_RN (m)")
ax[0].set_title("Resíduo vs latitude")

ax[1].scatter(rn_zet["lon"], rn_zet["resid"], s=20)
ax[1].set_xlabel("Longitude (°)")
ax[1].set_title("Resíduo vs longitude")

fig.suptitle("Diagnóstico de tendência espacial")
plt.tight_layout()
plt.savefig("../data/processed/validacao_zeta_trends.png", dpi=300)
plt.show()

plt.figure(figsize=(7.5,6))
sc = plt.scatter(outN["lon"], outN["lat"], c=outN["N_model"], s=25)
plt.colorbar(sc, label="N_model (m)")
plt.xlabel("Longitude (°)"); plt.ylabel("Latitude (°)")
plt.title("N do modelo nos pontos RN")
plt.tight_layout()
plt.savefig("../data/processed/N_model_RN_mapa.png", dpi=300)
plt.show()
